# Model Comparison Notebook

Evaluates multiple LLM configs (base model vs LoRA-adapted) across all 38 trading symbols over 2022–2025.
For each model × symbol, generates `N_SAMPLES` strategy candidates and scores them through the full reward pipeline.
Outputs per-model stats, top strategies per symbol, and cross-model comparison charts.

### Installations

In [ ]:
%%capture
import os, re
if 'COLAB_' not in ''.join(os.environ.keys()):
    %pip install unsloth
else:
    import torch; v = re.match(r'[0-9]{1,}\.[0-9]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + ('0.0.33.post1' if v=='2.9' else '0.0.32.post2' if v=='2.8' else '0.0.29.post3')
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

In [ ]:
%pip install plotly pandas -q

# Shared backtest / prompt / reward code: the `trading_rl` package in this repo
import os, sys
if os.path.isdir('../trading_rl'):   # local clone: import straight from the repo
    sys.path.insert(0, os.path.abspath('..'))
else:                                # Colab / Kaggle
    %pip install -q "trading-rl @ git+https://github.com/adhamhelmy/llm-fine-tuning.git"

### Secrets

In [ ]:
HF_TOKEN=''
ALPACA_API_KEY=''
ALPACA_SECRET_KEY=''

### Configuration

Edit `MODEL_CONFIGS` to add/remove models. Set `lora_adapter_path` to a HuggingFace repo ID or local path, or `None` for the base model.

In [ ]:
MODEL_CONFIGS = [
    {
        'label': 'Qwen2.5-32B base',
        'model_name': 'unsloth/Qwen2.5-Coder-32B-Instruct',
        'lora_adapter_path': None,
    },
    {
        'label': 'Qwen2.5-32B LoRA v1-500',
        'model_name': 'unsloth/Qwen2.5-Coder-32B-Instruct',
        'lora_adapter_path': 'adhamhelmy/qwen2.5-coder-32b-instruct-v1-500',
        'base_label': 'Qwen2.5-32B base',
    },
    {
        'label': 'Llama-3.1-8B base',
        'model_name': 'unsloth/Meta-Llama-3.1-8B-Instruct',
        'lora_adapter_path': None,
    },
    {
        'label': 'Llama-3.1-8B LoRA v1-500',
        'model_name': 'unsloth/Meta-Llama-3.1-8B-Instruct',
        'lora_adapter_path': 'adhamhelmy/meta-llama-3.1-8b-instruct-v1-500',
        'base_label': 'Llama-3.1-8B base',
    },
    {
        'label': 'Qwen2.5-7B base',
        'model_name': 'unsloth/Qwen2.5-Coder-7B-Instruct',
        'lora_adapter_path': None,
    },
    {
        'label': 'Qwen2.5-7B LoRA v1-500',
        'model_name': 'unsloth/Qwen2.5-Coder-7B-Instruct',
        'lora_adapter_path': 'adhamhelmy/qwen2.5-coder-7b-instruct-v1-500',
        'base_label': 'Qwen2.5-7B base',
    },
]

TEST_START = '2022-01-01'
TEST_END   = '2025-12-31'
N_SAMPLES  = 5   # strategy candidates per symbol per model

from trading_rl import TRAINING_SYMBOLS
SYMBOLS = TRAINING_SYMBOLS  # 38-symbol training universe

### Imports

In [ ]:
from trading_rl.model import Unsloth  # import first: unsloth must load before transformers

import os
import json

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

from trading_rl import Backtester, evaluate_symbol, make_prompt

### Evaluation Functions

In [ ]:
def evaluate_model(config, bt_instance, symbols=None, start=TEST_START, end=TEST_END, n_samples=N_SAMPLES):
    """Load model, evaluate all symbols, unload model, return flat list of result records."""
    symbols = symbols or SYMBOLS
    label = config['label']
    print(f"\n{'='*60}")
    print(f'Evaluating: {label}')
    print(f'Model: {config["model_name"]}')
    adapter = config.get('lora_adapter_path')
    print(f'LoRA adapter: {adapter or "None (base model)"}')
    print(f'Symbols: {len(symbols)}  |  Samples/symbol: {n_samples}  |  Period: {start} to {end}')
    print(f"{'='*60}")

    model = Unsloth(
        model_name=config['model_name'],
        max_seq_length=1024,
        lora_adapter_path=adapter,
        adapter_trainable=False,
        hf_token=HF_TOKEN,
    )

    records = []
    for sym in symbols:
        sym_recs = evaluate_symbol(model.generate, bt_instance, sym, start, end, n_samples,
                                   prompt=make_prompt(sym, start, end, compact=True))
        for r in sym_recs:
            r['model'] = label
        records.extend(sym_recs)

    model.unload()

    ok = sum(1 for r in records if r['status'] == 'profitable')
    print(f'\n[{label}] Done. Profitable: {ok}/{len(records)}')
    return records

### Run Evaluation

In [ ]:
# Initialize Backtrader singleton once — data cache persists across model switches
bt_instance = Backtester(ALPACA_API_KEY, ALPACA_SECRET_KEY, verbose=False)
bt_instance.load_bars(SYMBOLS, TEST_START, TEST_END)

In [ ]:
all_results = []
for config in MODEL_CONFIGS:
    records = evaluate_model(config, bt_instance)
    all_results.extend(records)

df = pd.DataFrame(all_results)
print(f'\nTotal samples: {len(df)}')
df.head(10)

In [ ]:
df.to_csv('model_comparison_results.csv', index=False)
print('Saved to model_comparison_results.csv')

### Results Analysis

In [ ]:
# Per-model summary statistics
summary = df.groupby('model').agg(
    total=('sample', 'count'),
    profitable=('status', lambda x: (x == 'profitable').sum()),
    loss=('status', lambda x: (x == 'loss').sum()),
    no_trades=('status', lambda x: (x == 'no_trades').sum()),
    invalid=('status', lambda x: x.isin(['missing_methods', 'invalid_code', 'exception', 'timeout']).sum()),
    mean_reward=('reward_score', 'mean'),
    median_reward=('reward_score', 'median'),
    mean_return=('return_pct', 'mean'),
    mean_sharpe=('sharpe_ratio', 'mean'),
    mean_annual=('avg_annual_return_pct', 'mean'),
).reset_index()
summary['profitable_pct'] = (summary['profitable'] / summary['total'] * 100).round(1)
display(summary.round(3))

In [ ]:
# Reward score distribution by model
fig = go.Figure()
for label in df['model'].unique():
    scores = df[df['model'] == label]['reward_score'].dropna()
    fig.add_trace(go.Box(
        y=scores, name=label,
        boxpoints='all', jitter=0.3, pointpos=-1.5,
    ))
fig.update_layout(
    title='Reward Score Distribution by Model',
    yaxis_title='Reward Score',
    template='plotly_dark',
)
fig.show()

In [ ]:
# Outcome breakdown (stacked bar)
status_counts = df.groupby(['model', 'status']).size().reset_index(name='count')
fig = px.bar(
    status_counts, x='model', y='count', color='status', barmode='stack',
    title='Outcome Breakdown by Model',
    color_discrete_map={
        'profitable': '#2ecc71', 'loss': '#e67e22', 'no_trades': '#95a5a6',
        'missing_methods': '#e74c3c', 'invalid_code': '#c0392b',
        'exception': '#9b59b6', 'timeout': '#7f8c8d',
    },
    template='plotly_dark',
)
fig.show()

In [ ]:
# Best reward per model per symbol (highest scoring sample)
best = (
    df[df['reward_score'].notna()]
    .sort_values('reward_score', ascending=False)
    .groupby(['model', 'symbol'])
    .first()
    .reset_index()
)

# Cross-model comparison: best reward per symbol
pivot = best.pivot_table(index='symbol', columns='model', values='reward_score')
melted = pivot.reset_index().melt(id_vars='symbol', var_name='model', value_name='best_reward')
fig = px.bar(
    melted, x='symbol', y='best_reward', color='model', barmode='group',
    title='Best Reward Score per Symbol — Model Comparison',
    template='plotly_dark',
)
fig.update_layout(xaxis_tickangle=45, height=500)
fig.show()

In [ ]:
# Return % heatmap per model
for label in df['model'].unique():
    sub = best[best['model'] == label][['symbol', 'return_pct']].set_index('symbol')
    fig = px.imshow(
        sub.T,
        title=f'Best Return % per Symbol — {label}',
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0,
        text_auto='.1f',
        template='plotly_dark',
    )
    fig.update_layout(height=200, xaxis_tickangle=45)
    fig.show()

In [ ]:
# Mean reward per model per symbol (heatmap)
for label in df['model'].unique():
    sub = df[df['model'] == label].groupby('symbol')['reward_score'].mean().reset_index()
    sub.columns = ['symbol', 'mean_reward']
    sub = sub.set_index('symbol')
    fig = px.imshow(
        sub.T,
        title=f'Mean Reward Score per Symbol — {label}',
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0,
        text_auto='.2f',
        template='plotly_dark',
    )
    fig.update_layout(height=200, xaxis_tickangle=45)
    fig.show()

In [ ]:
# Return delta: each LoRA vs its paired base model
pivot_ret = best.pivot_table(index='symbol', columns='model', values='return_pct')

for config in MODEL_CONFIGS:
    if not config.get('lora_adapter_path') or not config.get('base_label'):
        continue
    adapted_label = config['label']
    base_label    = config['base_label']
    if base_label not in pivot_ret.columns or adapted_label not in pivot_ret.columns:
        continue
    delta = (pivot_ret[adapted_label] - pivot_ret[base_label]).to_frame(name='delta_return_pct')
    fig = px.imshow(
        delta.T,
        title=f'Return Delta: {adapted_label} vs {base_label}',
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0,
        text_auto='.1f',
        template='plotly_dark',
    )
    fig.update_layout(height=200, xaxis_tickangle=45)
    fig.show()

In [ ]:
# Save best profitable strategy per model per symbol
save_dir = 'model_comparison_strategies'
os.makedirs(save_dir, exist_ok=True)
saved = 0

for _, row in best[best['status'] == 'profitable'].iterrows():
    model_dir = os.path.join(save_dir, row['model'].replace('/', '_').replace(' ', '_'))
    os.makedirs(model_dir, exist_ok=True)

    code = row.get('strategy_code')
    if code:
        with open(os.path.join(model_dir, f"{row['symbol']}_strategy.py"), 'w') as f:
            f.write(code)

    stats = {
        'model': row['model'],
        'symbol': row['symbol'],
        'return_pct': float(row['return_pct']) if pd.notna(row['return_pct']) else None,
        'sharpe_ratio': float(row['sharpe_ratio']) if pd.notna(row['sharpe_ratio']) else None,
        'avg_annual_return_pct': float(row['avg_annual_return_pct']) if pd.notna(row['avg_annual_return_pct']) else None,
        'max_drawdown_pct': float(row['max_drawdown_pct']) if pd.notna(row['max_drawdown_pct']) else None,
    }
    with open(os.path.join(model_dir, f"{row['symbol']}_stats.json"), 'w') as fj:
        json.dump(stats, fj, indent=2)
    saved += 1

print(f'Saved {saved} profitable strategies to {save_dir}/')